### Note
Inititally used MSFT and AAPL, but found that they had a very weak mean reversion, ADF failed, Sharpe was negative and a half-life of approximately 53 days,  that suggesting there was limited profitability and likely to have poor performance when trading after trading costs.
So the motivation for today is to find a pair of stocks with a stronger mean reversion.

In [34]:
# Updated prices.csv with more tickers

import yfinance as yf
import pandas as pd

tickers = ['AAPL','MSFT','NVDA','AMD','KO','PEP','XOM','CVX','JPM','BAC']

data = yf.download(tickers, start="2023-01-01", end="2026-02-20")['Close']
data.to_csv("../data/prices.csv")

[*********************100%***********************]  10 of 10 completed


,AAPL,AMD,BAC,CVX,JPM,KO,MSFT,NVDA,PEP,XOM
Date,,,,,,,,,,
2023-01-03,123.096024,64.019997,30.974270,151.901291,124.928711,57.519154,233.452820,14.300685,162.498108,95.434189
2023-01-04,124.365662,64.660004,31.556595,150.286133,126.093643,57.491737,223.240829,14.734250,162.099548,95.711967
2023-01-05,123.046814,62.330002,31.491901,152.992584,126.065750,56.833855,216.624466,14.250736,160.405838,97.853439
2023-01-06,127.574188,63.959999,31.806168,154.145020,128.478058,57.930328,219.177444,14.844140,164.028809,99.036171
2023-01-09,128.095840,67.239998,31.325510,152.940186,127.947151,57.208481,221.311462,15.612370,162.425613,97.190392


In [35]:

pairs = [
    ("AAPL","MSFT"),
    ("NVDA", "AMD"),
    ("KO", "PEP"),
    ("XOM", "CVX"),
    ("JPM", "BAC")
]

price_data = pd.read_csv("../data/prices.csv", index_col=0, parse_dates=True)
price_data.head()

,AAPL,AMD,BAC,CVX,JPM,KO,MSFT,NVDA,PEP,XOM
Date,,,,,,,,,,
2023-01-03,123.096024,64.019997,30.974270,151.901291,124.928711,57.519154,233.452820,14.300685,162.498108,95.434189
2023-01-04,124.365662,64.660004,31.556595,150.286133,126.093643,57.491737,223.240829,14.734250,162.099548,95.711967
2023-01-05,123.046814,62.330002,31.491901,152.992584,126.065750,56.833855,216.624466,14.250736,160.405838,97.853439
2023-01-06,127.574188,63.959999,31.806168,154.145020,128.478058,57.930328,219.177444,14.844140,164.028809,99.036171
2023-01-09,128.095840,67.239998,31.325510,152.940186,127.947151,57.208481,221.311462,15.612370,162.425613,97.190392


In [36]:
# Define a screening function
from statsmodels.tsa.stattools import coint
from statsmodels.api import OLS, add_constant
import numpy as np

def evaluate_pair(price_data, ticker1, ticker2):
    # Use log returns
    x = np.log(price_data[ticker1])
    y = np.log(price_data[ticker2])

    # Cointegration test
    score, p_value, _ = coint(y,x)

    # Hedge Ratio
    X = add_constant(x)
    model = OLS(y,X).fit()
    beta = model.params[1]

    # Spread
    spread  = y - beta * x

    # Estimate lambda
    spread_lag = spread.shift(1)
    spread_ret = spread - spread_lag
    spread_lag = spread_lag.dropna()
    spread_ret = spread_ret.loc[spread_lag.index]

    X_lag = add_constant(spread_lag)
    model_lag = OLS(spread_ret, X_lag).fit()
    lambda_ = model_lag.params.iloc[1]

    # Half-life
    half_life = -np.log(2)/ lambda_ if lambda_ < 0 else np.nan

    return{
        "pair": f"{ticker1} - {ticker2}",
        "coint_pvalue" : p_value,
        "lambda" : lambda_,
        "half_life" : half_life 
    }

In [37]:
# Test our pairs

results = []



for t1, t2 in pairs:
    res = evaluate_pair(price_data, t1, t2)
    results.append(res)

results_df = pd.DataFrame(results)
results_df

C:\Users\yogst\AppData\Local\Temp\ipykernel_49324\597926422.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta = model.params[1]
C:\Users\yogst\AppData\Local\Temp\ipykernel_49324\597926422.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta = model.params[1]
C:\Users\yogst\AppData\Local\Temp\ipykernel_49324\597926422.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta = model.params[1]
C:\Users\yogst\AppData\Local

,pair,coint_pvalue,lambda,half_life
0,AAPL - MSFT,0.807370,-0.007439,93.180327
1,NVDA - AMD,0.556731,-0.009519,72.814345
2,KO - PEP,0.314321,-0.017669,39.229750
3,XOM - CVX,0.037721,-0.026452,26.204279
4,JPM - BAC,0.082354,-0.019439,35.657666


### Intepretation
We have found only XOM and CVX are cointegrated, has a relatively fast half-life (twice as fast as AAPL-MSFT), therefore moving forward, we will be using XOM and CVX for backtesting.

Note that XOM (Exxon Mobil Corp) is a corporation that engages into the exploration and production of crude oil and natural gas, and CVX (Chevron Corp) engages in the integrated energy and chemical operations in United States and internationally (info from Yahoo Finance UK website)